In [16]:
2**12

4096

In [17]:
# scipy, convert "C:\Users\101119\Documents\Sound Recordings\Recording (5).m4a" to spectrogram
# To convert an audio file (like "Recording (5).m4a") to a spectrogram using scipy, you can follow these steps:
import scipy.io.wavfile as wav
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
# test and run ffmpeg to convert m4a to wav
import subprocess
import os
from pathlib import Path
import shutil

# subprocess.run(['ffmpeg', '-i', 'C:\\Users\\101119\\Documents\\Sound Recordings\\Recording (2).m4a', 'Recording.wav'])

In [18]:
# take "C:\Users\101119\Documents\Sound Recordings\Recording.m4a"
# split it into 2-second segments, with a stride of 1 second, and save each segment as real PCM .wav files.

import json
import math

def plot_spectrogram(audio_data, sample_rate, output_path):
    # Convert stereo/multi-channel audio to mono for spectrogram plotting
    audio_data = np.asarray(audio_data)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)

    # Generate the spectrogram
    # frequencies, times, Sxx = spectrogram(audio_data, fs=sample_rate)
    frequencies, times, Sxx = spectrogram(
    audio_data,
    fs=sample_rate,
    window='hann',
    nperseg=4096//4,
    noverlap=3840//4, #3072,
    scaling='spectrum',
    mode='magnitude'
                        )

    # Plot the spectrogram
    plt.figure(figsize=(20, 6))
    # output_path = os.path.join(
    #     r"D:\DSAI-teaching\brahmansh",
    #     os.path.splitext(os.path.basename(wav_path))[0] + ".png"
    # )

    plt.imsave(
        output_path,
        20 * np.log10(Sxx + 1e-12),
        origin="lower",
        cmap="viridis"      # optional
    )
    # plt.show()

    


def _get_audio_duration_seconds(path):
    result = subprocess.run(
        [
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'json',
            str(path),
        ],
        check=True,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    payload = json.loads(result.stdout)
    return float(payload['format']['duration'])

def split_save(path, segment_length=2, stride=1):
    src = Path(path)
    base_name = src.stem
    output_dir = Path.cwd() / f"{base_name}_segments"
    output_dir.mkdir(exist_ok=True)

    if segment_length <= 0 or stride <= 0:
        raise ValueError('segment_length and stride must be > 0')

    duration = _get_audio_duration_seconds(src)
    start_times = []
    t = 0.0
    # Generate overlapping windows with the requested stride.
    while t < duration-segment_length:
        start_times.append(t)
        t += stride

    for i, start in enumerate(start_times):
        out_file = output_dir / f"{base_name}_{i:03d}.wav"
        try:
            subprocess.run(
                [
                    'ffmpeg', '-y', '-nostdin', '-hide_banner', '-loglevel', 'error',
                    '-ss', f"{start:.3f}",
                    '-i', str(src),
                    '-t', str(segment_length),
                    # Force true WAV format that scipy can read.
                    '-acodec', 'pcm_s16le',
                    # '-ar', '16000',
                    # '-ac', '1',
                    str(out_file),
                ],
                check=True,
                stdin=subprocess.DEVNULL,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )
        except subprocess.CalledProcessError as e:
            print(f'ffmpeg failed for segment {i} starting at {start:.3f}s:')
            print(e.stderr)
            raise

    print(f"Generated {len(start_times)} segment(s) in: {output_dir}")
    return output_dir

In [19]:

def process_data(src_path, lst_prefixes):
    for prefix in lst_prefixes:
        print(f"processing files for {prefix}")
        for srcfile in os.listdir(src_path):
            if srcfile.lower().startswith(prefix):
                print(srcfile)
                wav_store = split_save(os.path.join(src_path, srcfile))
                for wavfile in os.listdir(wav_store):
                    # print(f"converting file: {wavfile}")
                    if wavfile.endswith('.wav'):
                        wav_path = os.path.join(wav_store, wavfile)
                        sample_rate, audio_data = wav.read(wav_path)
                        filename = wavfile.split(".")[0]
                        filepath = os.path.join(r"data\raw\\", prefix, filename)  +".png"
                        os.makedirs(os.path.join(r"data\raw\\", prefix), exist_ok=True)
                        plot_spectrogram(audio_data, sample_rate, filepath)
                shutil.rmtree(wav_store)



process_data(src_path=r"audiofiles", lst_prefixes=["ansh", "varsha"])



processing files for ansh
ansh_1.m4a
Generated 63 segment(s) in: d:\audio_classifier\ansh_1_segments


C:\Users\Brahmansh\AppData\Local\Temp\ipykernel_3672\967888783.py:26: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(20, 6))


ansh_10.m4a
Generated 52 segment(s) in: d:\audio_classifier\ansh_10_segments
ansh_11.m4a
Generated 56 segment(s) in: d:\audio_classifier\ansh_11_segments
ansh_2.m4a
Generated 49 segment(s) in: d:\audio_classifier\ansh_2_segments
ansh_3.m4a
Generated 43 segment(s) in: d:\audio_classifier\ansh_3_segments
ansh_4.m4a
Generated 44 segment(s) in: d:\audio_classifier\ansh_4_segments
ansh_5.m4a
Generated 52 segment(s) in: d:\audio_classifier\ansh_5_segments
ansh_6.m4a
Generated 67 segment(s) in: d:\audio_classifier\ansh_6_segments
ansh_7.m4a
Generated 63 segment(s) in: d:\audio_classifier\ansh_7_segments
ansh_8.m4a
Generated 49 segment(s) in: d:\audio_classifier\ansh_8_segments
ansh_9.m4a
Generated 43 segment(s) in: d:\audio_classifier\ansh_9_segments
processing files for varsha
varsha_1.m4a
Generated 55 segment(s) in: d:\audio_classifier\varsha_1_segments
varsha_2.m4a
Generated 54 segment(s) in: d:\audio_classifier\varsha_2_segments
varsha_3.m4a
Generated 60 segment(s) in: d:\audio_classifier

<Figure size 2000x600 with 0 Axes>

<Figure size 2000x600 with 0 Axes>

...

<Figure size 2000x600 with 0 Axes>

<Figure size 2000x600 with 0 Axes>

<Figure size 2000x600 with 0 Axes>

<Figure size 2000x600 with 0 Axes>

<Figure size 2000x600 with 0 Axes>

In [22]:

import os

d = {}

for root, dirs, files in os.walk("data"):
    count = sum(file.lower().endswith(".png") for file in files)

    if count > 0:
        d[root] = count

print(d)

{'data\\raw\\ansh': 581, 'data\\raw\\varsha': 466}
